## Paths

In [ ]:
from pathlib import Path

import sys

source_path = Path("./").resolve()
sys.path.append(str(source_path))

%load_ext autoreload
%autoreload 2


## Imports

In [1]:
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from src.back.data import split_dataset
from src.back.pipeline import *
from src.back.plots import plot_result
from src.back.preprocess import *
from src.back.whatif import *

## Load data

In [2]:
data_loader =  DataLoader(DB_PATH)
logger.info(f"Economics loaded: {data_loader.eco_anag['ECO_COD'].unique()}")
logger.info(f"Drivers loaded: {data_loader.driver_anag['DRV_COD'].unique()}")

[    INFO    ] Cutting 11 drivers due to high missing values
[    INFO    ] Cutting 28 eco due to high missing values
[    INFO    ] Economics loaded: ['ECO_001' 'ECO_003' 'ECO_005' 'ECO_006' 'ECO_007' 'ECO_008' 'ECO_009'
 'ECO_011' 'ECO_012' 'ECO_013' 'ECO_014' 'ECO_015' 'ECO_016' 'ECO_021'
 'ECO_022' 'ECO_023' 'ECO_024' 'ECO_025' 'ECO_026' 'ECO_027' 'ECO_029'
 'ECO_030' 'ECO_031' 'ECO_033' 'ECO_035' 'ECO_036' 'ECO_037' 'ECO_038'
 'ECO_039' 'ECO_040' 'ECO_041' 'ECO_042' 'ECO_043' 'ECO_044' 'ECO_045'
 'ECO_046' 'ECO_047' 'ECO_048' 'ECO_049' 'ECO_050' 'ECO_051' 'ECO_052'
 'ECO_053' 'ECO_054' 'ECO_055' 'ECO_056' 'ECO_057' 'ECO_058' 'ECO_059'
 'ECO_060' 'ECO_061' 'ECO_062' 'ECO_063' 'ECO_064' 'ECO_065' 'ECO_066'
 'ECO_067' 'ECO_068' 'ECO_069' 'ECO_070' 'ECO_071' 'ECO_072' 'ECO_073'
 'ECO_074' 'ECO_075' 'ECO_076' 'ECO_077' 'ECO_078' 'ECO_079' 'ECO_080'
 'ECO_081' 'ECO_082' 'ECO_083' 'ECO_084' 'ECO_086' 'ECO_087' 'ECO_088'
 'ECO_089' 'ECO_090' 'ECO_091' 'ECO_092' 'ECO_093' 'ECO_095' 'ECO_09

In [ ]:
check_drv_anag = data_loader.driver_anag
check_drv_sel = data_loader.driver_selected_df
check_eco_anag = data_loader.eco_anag
check_eco_best = data_loader.eco_best_model_df
check_eco_df = data_loader.eco_df
check_eco_forecast_df = data_loader.eco_forecast_df

check = check_drv_sel.merge(check_eco_best, on="ECO_COD", how="left")
check_group_fit_df = data_loader.eco_group_fit_df

In [18]:
from hierarchicalforecast.utils import aggregate
eco_df = data_loader.eco_df.copy()
eco_anag = data_loader.eco_anag.copy()
eco_anag["ECO_GRP_0"] = eco_anag["ECO_GRP_0"].astype(str)
eco_anag["ECO_GRP_1"] = eco_anag["ECO_GRP_1"].astype(str)
eco_anag["ECO_GRP_2"] = eco_anag["ECO_GRP_2"].astype(str)
eco_anag["ECO_GRP_2"] = eco_anag["ECO_GRP_2"].fillna('_')
eco_df = eco_df.merge(eco_anag, on="ECO_COD", how="inner").reset_index(drop=True)
eco_df['VALUE'] = eco_df['VALUE'].abs().fillna(0)
eco_grp, _, _ = aggregate(
        df=eco_df,
        spec=config.forecast.spec,
        time_col="DATE_RIF",
        target_cols=("VALUE",),
    )
cols = config.forecast.spec[-1]
eco_grp[cols] = eco_grp["unique_id"].str.split("/", expand=True)    
eco_grp.drop(columns=["unique_id"], inplace=True)
eco_grp1 = eco_grp.groupby(cols)["VALUE"].sum().reset_index().rename(columns={"VALUE": "VALUE_DET"})
cols2 = config.forecast.spec[-2]
eco_grp2 = eco_grp.groupby(cols2)["VALUE"].sum().reset_index().rename(columns={"VALUE": "VALUE_GRP"})
eco_grp1 = eco_grp1.merge(eco_grp2, on=cols2, how="left").reset_index(drop=True)
eco_grp1['WHEIGHT'] = eco_grp1['VALUE_DET'] / eco_grp1['VALUE_GRP']*100
eco_grp2 = eco_grp1[eco_grp1['WHEIGHT'] >= 2].reset_index(drop=True).sort_values(by=cols2 + ['WHEIGHT']).reset_index(drop=True)
ele_cod = eco_grp2['ECO_COD'].unique()

In [19]:
ele_cod

array(['ECO_321', 'ECO_326', 'ECO_324', 'ECO_328', 'ECO_319', 'ECO_325',
       'ECO_318', 'ECO_240', 'ECO_246', 'ECO_252', 'ECO_248', 'ECO_253',
       'ECO_257', 'ECO_250', 'ECO_249', 'ECO_191', 'ECO_241', 'ECO_244',
       'ECO_279', 'ECO_117', 'ECO_171', 'ECO_170', 'ECO_137', 'ECO_160',
       'ECO_110', 'ECO_104', 'ECO_109', 'ECO_334', 'ECO_163', 'ECO_333',
       'ECO_149', 'ECO_129', 'ECO_162', 'ECO_308', 'ECO_084', 'ECO_106',
       'ECO_277', 'ECO_111', 'ECO_212', 'ECO_209', 'ECO_261', 'ECO_289',
       'ECO_066', 'ECO_258', 'ECO_290', 'ECO_071', 'ECO_151', 'ECO_142',
       'ECO_157', 'ECO_158', 'ECO_195', 'ECO_268', 'ECO_267', 'ECO_042',
       'ECO_123', 'ECO_029', 'ECO_145', 'ECO_223', 'ECO_188', 'ECO_144',
       'ECO_045', 'ECO_130', 'ECO_043', 'ECO_238', 'ECO_089', 'ECO_239',
       'ECO_090', 'ECO_044', 'ECO_237', 'ECO_088', 'ECO_201', 'ECO_135',
       'ECO_332', 'ECO_138', 'ECO_198', 'ECO_064'], dtype=object)

## Reconcile

In [ ]:
forecasting_model_rec = ForecastingModel(configuration_path=ECONOMICS_MODEL_CONFIG_PATH, freq=config.forecast.freq)
df_rec, df_tot = fit_reconciliation(data_loader, forecasting_model_rec, aggregation_spec=config.forecast.spec)

In [ ]:
check_group_fit_df = data_loader.eco_group_fit_df

In [ ]:
data_loader.save_all()

In [5]:
check_whatif = data_loader.cause_effect_df

In [ ]:
# from hierarchicalforecast.utils import aggregate
# check_rec = data_loader.rec_forecast_df
# check_rec = check_rec[check_rec['ECO_COD']!=""].reset_index(drop=True)
# for cols in config.forecast.spec:
#     check_rec[cols] = check_rec[cols].fillna("")    
# df_aggr, _, _ = aggregate(
#     df=check_rec,
#     spec=config.forecast.spec,
#     time_col="DATE_RIF",
#     target_cols=("y_hat_rec",),
# )
# df_aggr[config.forecast.spec[-1]] = df_aggr['unique_id'].str.split('/', expand=True)
# for cols in config.forecast.spec:
#     df_aggr[cols] = df_aggr[cols].fillna("")    
# df_aggr = df_aggr.drop(columns=["unique_id"]).rename(columns={"y_hat_rec": "y_hat_rec_sum"})

# check_rec_sum = data_loader.rec_forecast_df.merge(df_aggr, on=["ECO_GRP_0", "ECO_GRP_1", "ECO_GRP_2", "ECO_COD", "DATE_RIF"], how="left")


In [ ]:
# data_loader.save_table('eco_anag', 'ECO_ANAG')

## Cause effect matrix what-if

In [13]:
cause_effect_matrix = calculate_cause_effect(data_loader)
cause_effect_matrix2 = cause_effect_matrix.copy()
cause_effect_matrix2['DRV_COD_RADIX'] = cause_effect_matrix['DRV_COD'].str[:7]
cause_effect_matrix2 = cause_effect_matrix2.drop(columns=["DRV_COD"])
cause_effect_matrix2 = cause_effect_matrix2.merge(data_loader.driver_anag[["DRV_COD", "DRV_DSC"]], left_on="DRV_COD_RADIX", right_on="DRV_COD", how="left")
cause_effect_matrix2 = cause_effect_matrix2.drop(columns=["DRV_COD"]).rename(columns={"DRV_DSC": "DRV_DSC_RADIX"})
cause_effect_matrix2 = cause_effect_matrix2.merge(data_loader.driver_anag[["DRV_COD", "DRV_DSC"]], left_on="DRV_COD_CE", right_on="DRV_COD", how="left")
cause_effect_matrix2 = cause_effect_matrix2.drop(columns=["DRV_COD"]).rename(columns={"DRV_DSC": "DRV_DSC_CE"})
cause_effect_matrix2 = cause_effect_matrix2[cause_effect_matrix2['COEF_CE'] != 0].reset_index(drop=True)
cause_effect_matrix2 = cause_effect_matrix2[["DRV_COD_CE", "DRV_COD_RADIX", "DRV_DSC_CE", "DRV_DSC_RADIX"]].drop_duplicates().reset_index(drop=True)

/home/anna/prj/ilabs-bankfcs/bankfcs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
/home/anna/prj/ilabs-bankfcs/bankfcs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
/home/anna/prj/ilabs-bankfcs/bankfcs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate

In [ ]:
# edits_list = ['DRV_017']
# cause_effect_matrix3 = data_loader.cause_effect_df.copy()
# cause_effect_matrix3['DRV_COD_RADIX'] = cause_effect_matrix3['DRV_COD'].str[:7]


# check = cause_effect_matrix3[(cause_effect_matrix3['DRV_COD_RADIX'].isin(edits_list)) & (cause_effect_matrix3['COEF_CE'] != 0)]['DRV_COD_CE'].unique()




In [ ]:
# edits_list = ['DRV_017', 'DRV_011']
# cause_effect_matrix3 = data_loader.cause_effect_df.copy()
# cause_effect_matrix3['DRV_COD_RADIX'] = cause_effect_matrix3['DRV_COD'].str[:7]

# found = set(edits_list)
# while True:
#     # Trova tutti i DRV_COD_CE collegati agli edits correnti
#     new_codes = set(
#         cause_effect_matrix3[
#             (cause_effect_matrix3['DRV_COD_RADIX'].isin(found)) &
#             (cause_effect_matrix3['COEF_CE'] != 0)
#         ]['DRV_COD_CE'].unique()
#     )
#     # print(new_codes)
#     # Solo i codici non già trovati
#     new_codes = new_codes - found
#     if not new_codes:
#         break
#     found.update(new_codes)

# # Alla fine, found contiene tutti i codici trovati iterativamente
# print(found)

In [14]:
data_loader.cause_effect_df = cause_effect_matrix.copy()
data_loader.save_all()

In [ ]:
# sim_df_long = propagate_shock(data_loader.drivers_forecast_df, cause_effect_matrix, "DRV_011", 6, 10000)
# check_whatif = data_loader.drivers_forecast_df.merge(sim_df_long, on=["DRV_COD", "DATE_RIF"], how="left")
# check_whatif['diff'] = check_whatif['FORECAST'] - check_whatif['FORECAST_WHATIF']